**[🏠 Course Home](00_START_HERE.ipynb)** | **🐍 [Python](../python/02_normal_conjugate_updating.ipynb)** | **🔵 [R](../r/02_normal_conjugate_updating.ipynb)** | **⬅️ [Prev: Ch 01: Discrete Beta-Binomial Conjugacy](01_foundations_and_conjugate_updating.ipynb)** | **➡️ [Next: Ch 03: Calculating a Mean (Frequentist)](03_calculating_a_mean_frequentist.ipynb)**

*🧭 Part I: Foundations, Conjugacy & Estimating a Mean — Chapter 02 of 22*

---

# Chapter 2: Continuous Conjugacy — Normal Models, Precision & The Gaussian Tug-of-War
### *Exact Analytical Updating for Continuous Gaussian Measurements*
### *& Inverse-Variance Weighting*

---

## 1. Course Context: Why We Need Continuous Conjugacy

In **Chapter 1**, we explored how beliefs rationally update when observing
discrete counts:
* Successes and failures of an automated test suite.
* Clicks versus non-clicks on a web user interface button.
* Heads versus tails on a sequence of coin flips.

Under the **Beta-Binomial conjugate model**, our initial belief acted as
"prior pebbles" placed into two physical buckets on a balance scale.
When real data arrived, updating our belief required nothing more than
elementary integer addition:

$$\alpha_{\text{post}} = \alpha_0 + k, \qquad \beta_{\text{post}} = \beta_0 + (n - k)$$

That simple pebble arithmetic made Bayesian learning effortless.

### The Limits of Discrete Counts
However, real-world science and engineering rarely present themselves as
tidy pass/fail tallies. Instead, physical systems produce **continuous
measurements along the real number line**:
* **Network packet latency** in milliseconds (e.g., $142.3\text{ ms}$,
  $185.7\text{ ms}$, $119.1\text{ ms}$).
* **Hardware processor core temperatures** in degrees Celsius (e.g.,
  $68.4^\circ\text{C}$, $74.1^\circ\text{C}$, $81.2^\circ\text{C}$).
* **Autonomous vehicle distances** to an obstacle measured by radar or
  sonar (e.g., $10.2\text{ m}$, $13.8\text{ m}$, $9.7\text{ m}$).
* **Battery voltage levels, audio frequencies, sensor calibrations, and
  financial returns**.

In these domains, the parameter we wish to estimate—the true mean $\mu$—is
not a bounded probability rate $p \in [0, 1]$. It is a continuous real
number $\mu \in (-\infty, \infty)$.

> **The Central Question of Chapter 2**:  
> *When our prior belief is a continuous bell curve (Normal distribution) and
> our incoming physical measurement is also Gaussian, how does Bayes' Rule
> update our belief?*

The miraculous answer is **Normal-Normal Conjugacy**. Just as the
Beta-Binomial model turns discrete updates into addition, Gaussian conjugacy
turns continuous calculus into an elegant, intuitive **weighted average**!

---

## 2. The Physical Analogy: Two GPS Sensors & The Spring-Loaded Tug-of-War

Before touching any algebraic formulas, let us build a physical mental model
that will stay with you for the rest of your engineering career.

Imagine you are designing the navigation system for an autonomous mobile
robot. The vehicle is attempting to estimate its true distance $\mu$ to a
docking wall along a straight track.

You have two competing sources of information:
1. **The Prior State of Knowledge ($p(\mu)$)**: An estimate from an
   onboard ultrasonic sensor or dead-reckoning filter, centered at
   $\mu_0$ with uncertainty standard deviation $\sigma_0$ (variance
   $\sigma_0^2$).
2. **The Observed Data Measurement ($p(y \mid \mu)$)**: A new reading $y$
   from an optical LIDAR sensor with measurement noise standard deviation
   $\sigma$ (variance $\sigma^2$).

Both sources are noisy. Neither is perfect. How should Bayes' Rule combine
them into an updated posterior belief $\mu_{\text{post}}$?

### Variance vs. Precision: Noise vs. Stiffness
To understand the compromise, we must clarify what variance means
physically:
* **Variance ($\sigma^2$) is noise, sloppiness, and scatter.**  
  A sensor with a large variance is loose, erratic, and uncertain. Its
  probability bell curve is wide, low, and blurry.
* **Precision ($\tau = 1/\sigma^2$) is information stiffness and voting
  power!**  
  In classical physics and statistics, the reciprocal of variance is called
  **Precision** (denoted $\tau$):
  $$\tau = \frac{1}{\sigma^2}$$
  If variance is how sloppy a sensor is, precision is how **stiff,
  reliable, and authoritative** it is.

> [!IMPORTANT]
> **The Golden Law of Gaussian Inference: Voting Power = Precision**  
> In Gaussian inference, every information source casts votes in proportion
> to its precision (inverse variance).  
> A sensor with half the noise standard deviation has **four times the
> precision** ($1/1^2 = 4 \times 1/2^2$), granting it **four times the
> voting power** in deciding the posterior estimate!

### The Mechanical Tug-of-War: Springs on a Frictionless Slider
Imagine a frictionless slider sitting on a measuring rail. The position of
the slider represents our best estimate of the parameter $\mu$.

Now attach two mechanical springs to opposite sides of the slider:
* **Spring 0 (The Prior)** is anchored firmly at position $\mu_0$. Its
  mechanical stiffness (spring constant) is $k_0 = \tau_0 = 1/\sigma_0^2$.
* **Spring L (The Measurement / Likelihood)** is anchored firmly at
  position $y$. Its mechanical stiffness is $k_L = \tau_{\text{data}} =
  1/\sigma^2$.

```text
                     THE SPRING-LOADED GAUSSIAN TUG-OF-WAR
 
       Prior Anchor                                       Data Measurement
      (Position mu_0)                                       (Position y)
            │                                                    │
            ▼                                                    ▼
          [mu_0] ======/\/\/\/\/\====== [mu_post] ======/\/\/\/\/\====== [y]
                     Spring 0 (tau_0)    Slider       Spring L (tau_data)
                       (Stiffness)     Equilibrium        (Stiffness)
 
    Net Force Balance:  tau_0 * (mu_post - mu_0) = tau_data * (y - mu_post)
    Total Stiffness:    tau_post = tau_0 + tau_data  (Precisions Simply Add!)
```

When you release the slider, the two springs pull against each other:
* Spring 0 exerts a Hooke's Law restoring force pulling toward $\mu_0$:
  $$F_0 = \tau_0 (\mu_0 - \mu)$$
* Spring L exerts a restoring force pulling toward $y$:
  $$F_L = \tau_{\text{data}} (y - \mu)$$

Where does the slider come to rest? At mechanical equilibrium, the opposing
forces must cancel out exactly ($F_0 + F_L = 0$):

$$\tau_0 (\mu_{\text{post}} - \mu_0) + \tau_{\text{data}} (\mu_{\text{post}} - y) = 0$$

$$(\tau_0 + \tau_{\text{data}}) \mu_{\text{post}} = \tau_0 \mu_0 + \tau_{\text{data}} y$$

$$\mathbf{\mu_{\text{post}} = \frac{\tau_0 \mu_0 + \tau_{\text{data}} y}{\tau_0 + \tau_{\text{data}}}}$$

### What Happens to the Total Stiffness?
Because the two springs pull on the slider **in parallel**, their mechanical
stiffnesses simply combine:

$$\mathbf{\tau_{\text{post}} = \tau_0 + \tau_{\text{data}}}$$

The combined system is **mechanically stiffer than either individual
spring**! If you attempt to displace the slider from its resting position,
you must fight against the tension of *both* springs simultaneously.

Because precision is stiffness and variance is looseness, the posterior
variance is simply the reciprocal of the total stiffness:

$$\mathbf{\sigma_{\text{post}}^2 = \frac{1}{\tau_{\text{post}}} = \frac{1}{\tau_0 + \tau_{\text{data}}} = \frac{1}{\frac{1}{\sigma_0^2} + \frac{1}{\sigma^2}}}$$

The physical mechanics of springs and the rational mathematics of Bayes'
Rule produce the exact same outcome!

---

## 3. The Mathematical Miracle: Completing the Square (Without the Pain)

Let us now examine why the mathematics of probability produces this
identical spring-like result.

According to Bayes' Rule:

$$P(\mu \mid y) \propto P(y \mid \mu) \cdot P(\mu)$$

### Step A: Write the Prior and Likelihood as Gaussians
The prior distribution over the unknown mean $\mu$ is a Normal distribution
with mean $\mu_0$ and variance $\sigma_0^2 = 1/\tau_0$:

$$P(\mu) = \frac{1}{\sqrt{2\pi\sigma_0^2}} \exp\left(-\frac{\tau_0}{2}(\mu - \mu_0)^2\right) \propto \exp\left(-\frac{\tau_0}{2}(\mu - \mu_0)^2\right)$$

The likelihood of observing measurement $y$ given true mean $\mu$ with
instrument noise variance $\sigma^2 = 1/\tau_{\text{data}}$ is:

$$P(y \mid \mu) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left(-\frac{\tau_{\text{data}}}{2}(y - \mu)^2\right) \propto \exp\left(-\frac{\tau_{\text{data}}}{2}(\mu - y)^2\right)$$

### Step B: Multiply and Add Exponents
When we multiply two exponential functions, their exponents add together:

$$P(\mu \mid y) \propto \exp\left(-\frac{1}{2} \left[ \tau_0(\mu - \mu_0)^2 + \tau_{\text{data}}(\mu - y)^2 \right] \right)$$

Look inside the square brackets. We have the sum of two quadratic parabolas.
Let us expand both squared terms:

$$\tau_0(\mu^2 - 2\mu\mu_0 + \mu_0^2) + \tau_{\text{data}}(\mu^2 - 2\mu y + y^2)$$

Now group the terms by powers of the unknown parameter $\mu$:

$$= \underbrace{(\tau_0 + \tau_{\text{data}})}_{\text{Leading coefficient of } \mu^2} \mu^2 - 2 \underbrace{(\tau_0 \mu_0 + \tau_{\text{data}} y)}_{\text{Linear coefficient of } \mu} \mu + \underbrace{(\tau_0 \mu_0^2 + \tau_{\text{data}} y^2)}_{\text{Constant with respect to } \mu}$$

### Step C: Identify the Posterior Precision
In standard Gaussian form, any distribution over $\mu$ with mean $m$ and
precision $\tau$ has the quadratic exponent:

$$-\frac{\tau}{2}(\mu - m)^2 = -\frac{1}{2} \left( \tau \mu^2 - 2\tau m \mu + \tau m^2 \right)$$

Notice that the coefficient multiplying $\mu^2$ is **always the precision
$\tau$**!

Looking at our expanded exponent, the coefficient multiplying $\mu^2$ is
simply $\tau_0 + \tau_{\text{data}}$. Therefore, without solving any
integrals:

$$\mathbf{\tau_{\text{post}} = \tau_0 + \tau_{\text{data}}}$$

**The precisions simply add!**

### Step D: Complete the Square for the Posterior Mean
To find the center of the new bell curve, factor out the total precision
$\tau_{\text{post}}$:

$$(\tau_0 + \tau_{\text{data}}) \left[ \mu^2 - 2 \left( \frac{\tau_0 \mu_0 + \tau_{\text{data}} y}{\tau_0 + \tau_{\text{data}}} \right) \mu \right]$$

Completing the square inside the bracket gives:

$$= \tau_{\text{post}} \left( \mu - \mu_{\text{post}} \right)^2 + C$$

where:

$$\mathbf{\mu_{\text{post}} = \frac{\tau_0 \mu_0 + \tau_{\text{data}} y}{\tau_0 + \tau_{\text{data}}}}$$

and $C$ is a constant that does not involve $\mu$. When passed through the
exponential, $\exp(C)$ becomes a scaling factor that is absorbed by the
normalizing constant.

> [!TIP]
> **The Conjugate Magic of the Normal Distribution**  
> 1. Multiplying two Gaussian bell curves produces **another perfect Gaussian
>    bell curve**!
> 2. The posterior precision is the **sum of the precisions**:
>    $\tau_{\text{post}} = \tau_0 + \tau_{\text{data}}$.
> 3. The posterior mean is the **precision-weighted average**:
>
> $$\mu_{\text{post}} = \frac{\tau_0 \mu_0 + \tau_{\text{data}} y}{\tau_0 + \tau_{\text{data}}}$$

### Formal Proof: Why Uncertainty Strictly Decreases
A fundamental question in data science is: *Can observing new data ever make
a Gaussian posterior more uncertain than the prior was?*

The answer is **never**. Combining two noisy Gaussian measurements always
produces a posterior with strictly smaller variance than either source
alone:

1. Prior precision $\tau_0 = 1/\sigma_0^2 > 0$ and data precision
   $\tau_{\text{data}} = 1/\sigma^2 > 0$ are both strictly positive.
2. Therefore, their sum is strictly greater than each individual component:
   $$\tau_{\text{post}} = \tau_0 + \tau_{\text{data}} > \tau_0$$
   $$\tau_{\text{post}} = \tau_0 + \tau_{\text{data}} > \tau_{\text{data}}$$
3. Taking the reciprocal of both inequalities (which reverses the inequality
   sign for positive numbers):
   $$\sigma_{\text{post}}^2 = \frac{1}{\tau_{\text{post}}} < \frac{1}{\tau_0} = \sigma_0^2$$
   $$\sigma_{\text{post}}^2 = \frac{1}{\tau_{\text{post}}} < \frac{1}{\tau_{\text{data}}} = \sigma^2$$
4. Taking the square root:
   $$\mathbf{\sigma_{\text{post}} < \min(\sigma_0, \sigma)}$$

$\blacksquare$

Every independent physical measurement—no matter how noisy—adds positive
precision to the system, tightening the bounds of our knowledge.

---

## 4. Step-by-Step Worked Numerical Example: Autonomous Vehicle Sensor Fusion

Let us walk through a complete, concrete numerical calculation for our
autonomous vehicle navigation problem.

### The Problem Setup
An autonomous mobile robot is docking against a loading bay wall:
* **Prior Belief from Sonar**: An ultrasonic acoustic sensor reports an
  initial distance estimate of $\mu_0 = 10.0\text{ meters}$ with standard
  deviation $\sigma_0 = 2.0\text{ meters}$ (prior variance $\sigma_0^2 =
  4.0\text{ m}^2$).
* **New Measurement from Optical LIDAR**: A pulsed laser rangefinder
  captures a fresh measurement of $y = 14.0\text{ meters}$ with instrument
  noise standard deviation $\sigma_{\text{lidar}} = 1.0\text{ meter}$
  (variance $\sigma_{\text{lidar}}^2 = 1.0\text{ m}^2$).

Notice the tension: the sonar sensor says the wall is 10 meters away, but the
laser says it is 14 meters away. Where should our fused estimate lie?

### Step 1: Compute the Individual Precisions
First, convert the noise variances into stiffness weights (precisions):

$$\tau_0 = \frac{1}{\sigma_0^2} = \frac{1}{2.0^2} = \frac{1}{4.0} = \mathbf{0.25\text{ m}^{-2}}$$

$$\tau_{\text{lidar}} = \frac{1}{\sigma_{\text{lidar}}^2} = \frac{1}{1.0^2} = \frac{1}{1.0} = \mathbf{1.00\text{ m}^{-2}}$$

Notice that the LIDAR has **4 times the precision** of the Sonar! Why?
Because the LIDAR standard deviation is half as large ($1.0\text{ m}$ vs.
$2.0\text{ m}$), and precision scales inversely with the *square* of the
error: $(2/1)^2 = 4$.

### Step 2: Compute the Total Posterior Precision
Precisions simply add:

$$\tau_{\text{post}} = \tau_0 + \tau_{\text{lidar}} = 0.25 + 1.00 = \mathbf{1.25\text{ m}^{-2}}$$

### Step 3: Compute the Voting Shares
Calculate each source's percentage share of the total vote:

$$W_{\text{prior}} = \frac{\tau_0}{\tau_{\text{post}}} = \frac{0.25}{1.25} = \mathbf{0.20 \quad (20\% \text{ vote})}$$

$$W_{\text{lidar}} = \frac{\tau_{\text{lidar}}}{\tau_{\text{post}}} = \frac{1.00}{1.25} = \mathbf{0.80 \quad (80\% \text{ vote})}$$

The LIDAR commands $80\%$ of the compromise, while the fuzzier Sonar
controls only $20\%$.

### Step 4: Compute the Posterior Mean
Compute the precision-weighted average:
>
> $$\mu_{\text{post}} = \frac{\tau_0 \mu_0 + \tau_{\text{data}} y}{\tau_0 + \tau_{\text{data}}}$$

$$\mu_{\text{post}} = W_{\text{prior}} \mu_0 + W_{\text{lidar}} y$$

$$\mu_{\text{post}} = (0.20 \times 10.0\text{ m}) + (0.80 \times 14.0\text{ m})$$

$$\mu_{\text{post}} = 2.0\text{ m} + 11.2\text{ m} = \mathbf{13.2\text{ meters}}$$

The estimate sits at $13.2\text{ m}$—massively pulled toward the sharper
LIDAR reading ($14.0\text{ m}$), but still respecting the sonar's input.

### Step 5: Compute the Posterior Uncertainty
Convert the posterior precision back into variance and standard deviation:

$$\sigma_{\text{post}}^2 = \frac{1}{\tau_{\text{post}}} = \frac{1}{1.25} = \mathbf{0.80\text{ m}^2}$$

$$\sigma_{\text{post}} = \sqrt{0.80\text{ m}^2} \approx \mathbf{0.8944\text{ meters}}$$

### Summary of the Fusion Result
Look at the progression of uncertainty:
* **Prior Sonar Uncertainty**: $\sigma_0 = 2.000\text{ m}$
* **LIDAR Instrument Noise**: $\sigma_{\text{lidar}} = 1.000\text{ m}$
* **Fused Posterior Uncertainty**: $\sigma_{\text{post}} \approx 0.894\text{ m}$

Even though the Sonar reading was relatively noisy, incorporating it into the
LIDAR data **tightened the final uncertainty by over $10\%$** below that of
the LIDAR alone ($0.894 < 1.000$). No information was wasted.

---

## 5. The Three Archetypes of Bayesian Updating

By adjusting the relative stiffness of the prior ($\tau_0$) versus the
data ($\tau_{\text{data}}$), we observe the three universal archetypes of
rational inference:

### Archetype 1: The Uninformative Prior (Data Dominates)
* **Mathematical Condition**: $\tau_0 \to 0$ (or prior variance
  $\sigma_0^2 \to \infty$).
* **Physical Picture**: Spring 0 is made of loose cooked spaghetti. It has
  virtually zero restoring force.
* **The Mathematical Outcome**:
  $$W_{\text{prior}} \to 0, \qquad W_{\text{data}} \to 1$$
  $$\mu_{\text{post}} \to y, \qquad \sigma_{\text{post}} \to \sigma$$
* **Engineering Meaning**: When you know nothing beforehand, you let the
  data dictate your belief. This directly recovers standard **frequentist
  Maximum Likelihood Estimation (MLE)** as a special sub-case of Bayesian
  inference!

### Archetype 2: The Stubborn Prior (Bayesian Regularization & Shrinkage)
* **Mathematical Condition**: $\tau_0 \gg \tau_{\text{data}}$ (prior
  precision vastly exceeds data precision).
* **Physical Picture**: Spring 0 is a rigid steel I-beam, while Spring L is a
  fragile rubber band.
* **The Mathematical Outcome**:
  $$W_{\text{prior}} \approx 1, \qquad W_{\text{data}} \approx 0$$
  $$\mu_{\text{post}} \approx \mu_0, \qquad \sigma_{\text{post}} \approx \sigma_0$$
* **Engineering Meaning**: When historical baseline telemetry is backed by
  millions of runs, a single noisy or anomalous sensor reading cannot easily
  derail your system. The prior **regularizes and shrinks** the estimate
  toward the established baseline, preventing knee-jerk false alarms.

### Archetype 3: Equal Partners (Symmetric Compromise)
* **Mathematical Condition**: $\tau_0 = \tau_{\text{data}}$ (equal
  stiffness, meaning $\sigma_0 = \sigma$).
* **Physical Picture**: Two identical springs pull with equal mechanical
  advantage from opposite sides.
* **The Mathematical Outcome**:
  $$W_{\text{prior}} = W_{\text{data}} = 0.50$$
  $$\mu_{\text{post}} = \frac{\mu_0 + y}{2} \quad \text{(Exact Midpoint)}$$
  $$\tau_{\text{post}} = 2\tau_0 \implies \sigma_{\text{post}}^2 = \frac{\sigma^2}{2} \implies \mathbf{\sigma_{\text{post}} = \frac{\sigma}{\sqrt{2}} \approx 0.707 \sigma}$$
* **Engineering Meaning**: Combining two equally reliable measurements
  always splits the difference and **reduces error standard deviation by
  $\approx 29.3\%$** ($1 - 1/\sqrt{2}$).

---

## 6. Scaling to $N$ Observations & The Bernstein-von Mises Theorem

What happens when our sensor does not record just one solitary data point,
but a continuous stream of $n$ independent physical measurements
$y_1, y_2, \dots, y_n$?

Let the sample average of these $n$ observations be $\bar{y} = \frac{1}{n}
\sum_{i=1}^n y_i$, with known measurement noise $\sigma^2$.

### The Multi-Observation Update Rules
From elementary statistics, the sample mean $\bar{y}$ has sampling
variance $\sigma^2 / n$. Therefore, the total information stiffness
provided by the data is:

$$\tau_{\text{data}} = \frac{1}{\sigma^2 / n} = \frac{n}{\sigma^2}$$

Notice that **data precision grows strictly linearly with sample size $n$**!

Plugging this into our Gaussian conjugate formulas gives the general
multi-observation updating laws:

$$\mathbf{\tau_{\text{post}} = \tau_0 + \frac{n}{\sigma^2}}$$

$$\mathbf{\mu_{\text{post}} = \frac{\tau_0 \mu_0 + \left(\frac{n}{\sigma^2}\right) \bar{y}}{\tau_0 + \frac{n}{\sigma^2}} = \left(\frac{\tau_0}{\tau_0 + \frac{n}{\sigma^2}}\right) \mu_0 + \left(\frac{\frac{n}{\sigma^2}}{\tau_0 + \frac{n}{\sigma^2}}\right) \bar{y}}$$

$$\mathbf{\sigma_{\text{post}}^2 = \frac{1}{\tau_0 + \frac{n}{\sigma^2}} = \frac{1}{\frac{1}{\sigma_0^2} + \frac{n}{\sigma^2}}}$$

### The Bernstein-von Mises Theorem: The Inevitable Triumph of Evidence
Examine what happens mathematically as the sample size grows large
($n \to \infty$):
1. The data precision $n / \sigma^2$ increases without bound ($n / \sigma^2
   \to \infty$).
2. The prior precision $\tau_0$ remains a fixed, finite constant.
3. The voting share awarded to the prior decays to zero:
   $$\lim_{n \to \infty} W_{\text{prior}} = \lim_{n \to \infty} \frac{\tau_0}{\tau_0 + n/\sigma^2} = 0$$
4. The voting share awarded to the empirical evidence approaches $100\%$:
   $$\lim_{n \to \infty} W_{\text{data}} = \lim_{n \to \infty} \frac{n/\sigma^2}{\tau_0 + n/\sigma^2} = 1$$
5. The posterior mean locks onto the empirical sample mean:
   $$\lim_{n \to \infty} \mu_{\text{post}} = \bar{y}$$
6. The posterior standard deviation collapses to zero at the classical
   rate:
   $$\lim_{n \to \infty} \sigma_{\text{post}} = \frac{\sigma}{\sqrt{n}}$$

In mathematical statistics, this convergence is codified by the
**Bernstein-von Mises Theorem**:

> [!NOTE]
> **The Bernstein-von Mises Theorem (The Triumph of Data)**  
> As the volume of empirical data increases, the influence of the prior
> distribution fades to zero.  
> Two scientists who begin with wildly different subjective priors—provided
> neither assigned zero probability to the truth—will inevitably converge to
> the exact same posterior belief once sufficient evidence has been
> collected!

---

## 7. The Luxury and The Catch: When Conjugacy Breaks Down

Gaussian conjugacy is an intellectual masterpiece. It allows engineers to
fuse sensor streams and telemetry in real time on microcontrollers using
simple arithmetic addition, completely bypassing multidimensional numerical
integration.

### The Catch
However, this analytical luxury rests on two fragile assumptions:
1. **Known, Fixed Noise ($\sigma$)**: We assumed that the measurement noise
   variance $\sigma^2$ is perfectly known in advance. In the real world,
   sensor noise is often unknown and must be estimated simultaneously with
   the mean $\mu$.
2. **Strict Bell Curve Shapes**: We assumed that both the prior belief and
   the physical noise follow ideal Gaussian bell curves. If latency data has
   heavy exponential tails, or sensor data suffers from multimodal glitches,
   the Gaussian conjugacy assumption fails.

The moment both the mean $\mu$ and the variance $\sigma^2$ are unknown, or
when the physical process deviates from ideal textbook shapes, **analytical
conjugacy breaks down completely**.

What do we do when we can no longer solve Bayes' Rule with pen and paper?
* In **Chapter 5**, we explore **Grid Approximation** to evaluate arbitrary
  posteriors over a coordinate mesh.
* In **Chapter 6** and **Chapter 7**, we explore **Laplace Approximation**
  to fit curvature domes over the mode using the Hessian matrix.
* In **Chapters 8, 9, and 10**, we unleash **Markov Chain Monte Carlo
  (MCMC)** to explore high-dimensional parameter spaces without making
  any textbook conjugate assumptions!

---


---
## 🧭 Course Navigation

* 🏠 **[Course Home & Master Syllabus](00_START_HERE.ipynb)**
* ⬅️ **Previous**: **[Ch 01: Discrete Beta-Binomial Conjugacy](01_foundations_and_conjugate_updating.ipynb)** *(Part I: Foundations, Conjugacy & Estimating a Mean)*
* ➡️ **Next**: **[Ch 03: Calculating a Mean (Frequentist)](03_calculating_a_mean_frequentist.ipynb)** *(Part I: Foundations, Conjugacy & Estimating a Mean)*
* 🔄 **Companion Tracks for Chapter 02**: [📖 Conceptual](../conceptual/02_normal_conjugate_updating.ipynb) | [🐍 Python](../python/02_normal_conjugate_updating.ipynb) | [🔵 R](../r/02_normal_conjugate_updating.ipynb)
